# Fine-tune Trung → Việt trên Colab (3 giờ/lần train)

[Open in Colab](https://colab.research.google.com/github/22imer/AIChallenge77/blob/main/finetune_colab.ipynb)

Dùng **Helsinki-NLP/opus-mt-zh-vi**: Marian chuyên dịch Trung–Việt, giấy phép Apache-2.0.
Chọn checkpoint nhỏ chuyên một cặp ngôn ngữ thay vì model đa ngôn ngữ lớn để ưu tiên dễ chạy.
Không bổ sung dữ liệu ngoài để fine-tune; pretrained weights vốn đã học từ dữ liệu của tác giả model.
Giữ nguyên khoảng trắng và dấu `_` của dữ liệu cuộc thi. BLEU dưới đây là **metric nội bộ**, chưa xác nhận metric chính thức.

Chọn runtime GPU. Chuẩn bị `MyDrive/AIChallenge77/datasets/dataset.zip`, rồi Run All.
Sửa `ROOT` nếu Drive đang dùng đường dẫn khác. Tất cả dữ liệu/tokenized batches chạy ở `/content`.
Mỗi lần chạy cell training có ngân sách 3 giờ, dành 10 phút cuối để lưu và dừng.
Đây là giới hạn mềm: một GPU operation hoặc thao tác Drive đang chạy không thể bị ngắt an toàn đúng từng giây.
Download, đánh giá pretrained ban đầu và xuất submission nằm ngoài ngân sách training.
Resume giữ nguyên thứ tự batch, optimizer, scaler và RNG; có thể mất công việc từ lần lưu định kỳ gần nhất (~10 phút).
Không chạy hai phiên cùng `RUN_NAME` và không load checkpoint từ nguồn không tin cậy.


In [ ]:
# %pip install -q transformers==4.57.3 sentencepiece==0.2.1 sacrebleu==2.5.1 sacremoses==0.1.1 pandas==2.2.3
# Không reinstall PyTorch GPU của Colab. Nếu vừa đổi version trong phiên đã import,
# restart runtime trước khi chạy tiếp.
# Cell cài dependency đang tắt theo bản Colab. Với runtime mới, bỏ dấu # ở dòng %pip rồi chạy cell này.


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
import uuid
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import MarianMTModel, MarianTokenizer, DataCollatorForSeq2Seq


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AIChallenge77')
RUN_NAME = 'marian-zh-vi-none-v1'
RUN_DIR = ROOT / 'checkpoints' / RUN_NAME
SUBMISSION_DIR = ROOT / 'submissions' / RUN_NAME
LOCAL_DIR = Path('/content/ai77_finetune')
for directory in (RUN_DIR, SUBMISSION_DIR, LOCAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Helsinki-NLP/opus-mt-zh-vi'
MODEL_REVISION = '67ea2dbfbaf13a16772a40346d3d72b59e591443'
SEED = 42
VALID_FRACTION = 0.10
MAX_SOURCE_TOKENS = 256
MAX_TARGET_TOKENS = 256
NUM_BEAMS = 4
LEARNING_RATE = 2e-5
MAX_EPOCHS = 10
TRAIN_SECONDS = 3 * 60 * 60
SAVE_RESERVE_SECONDS = 10 * 60
SAVE_EVERY_SECONDS = 10 * 60
TEST_SPLIT = 'public_test'  # Đổi thành 'private_test' khi cần nộp private.

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU trong Runtime > Change runtime type trước khi train.')
DEVICE = torch.device('cuda')
AMP = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Versions:', torch.__version__, transformers.__version__, sacrebleu.__version__)


In [ ]:
import random
from pathlib import Path


def read_parallel(src_path, tgt_path):
    # Read together: dropping blanks independently shifts all subsequent pairs.
    src = Path(src_path).read_text(encoding='utf-8-sig').splitlines()
    tgt = Path(tgt_path).read_text(encoding='utf-8-sig').splitlines()
    if len(src) != len(tgt):
        raise ValueError(f'Unaligned parallel files: {len(src)} source vs {len(tgt)} target')
    pairs = [(s.strip(), t.strip()) for s, t in zip(src, tgt) if s.strip() and t.strip()]
    pairs = list(dict.fromkeys(pairs))
    if not pairs:
        raise ValueError('No usable translation pairs')
    return pairs


def split_pairs(pairs, valid_fraction=0.1, seed=42):
    sources = sorted({s for s, _ in pairs})
    if len(sources) < 2 or not 0 < valid_fraction < 1:
        raise ValueError('Need at least two source groups and a fraction between 0 and 1')
    random.Random(seed).shuffle(sources)
    size = min(len(sources) - 1, max(1, round(len(sources) * valid_fraction)))
    valid_sources = set(sources[:size])
    train = [(s, t) for s, t in pairs if s not in valid_sources]
    valid = [(s, t) for s, t in pairs if s in valid_sources]
    return train, valid


In [ ]:
archive = ROOT / 'datasets' / 'dataset.zip'
local_zip = LOCAL_DIR / 'dataset.zip'
shutil.copy2(archive, local_zip)
with zipfile.ZipFile(local_zip) as z:
    # Extract only the files used by this run, without trusting archive paths.
    for member in ('dataset/train/train.zh', 'dataset/train/train.vi',
                   f'dataset/{TEST_SPLIT}/{TEST_SPLIT}.zh'):
        destination = LOCAL_DIR / member
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(z.read(member))

src_path = LOCAL_DIR / 'dataset/train/train.zh'
tgt_path = LOCAL_DIR / 'dataset/train/train.vi'
pairs = read_parallel(src_path, tgt_path)
train_pairs, valid_pairs = split_pairs(pairs, VALID_FRACTION, SEED)
# Never discard blank test lines or reorder the test set.
test_src = (LOCAL_DIR / f'dataset/{TEST_SPLIT}/{TEST_SPLIT}.zh').read_text(
    encoding='utf-8-sig').splitlines()
print('Usable unique pairs:', len(pairs), 'train:', len(train_pairs), 'validation:', len(valid_pairs))
print('Test rows:', len(test_src))
assert not ({s for s, _ in train_pairs} & {s for s, _ in valid_pairs})

contract = {
    'model': MODEL_ID, 'revision': MODEL_REVISION,
    'data_sha256': hashlib.sha256(src_path.read_bytes() + b'\x00' + tgt_path.read_bytes()).hexdigest(),
    'seed': SEED, 'valid_fraction': VALID_FRACTION,
    'source_limit': MAX_SOURCE_TOKENS, 'target_limit': MAX_TARGET_TOKENS,
    'learning_rate': LEARNING_RATE, 'beams': NUM_BEAMS,
    'transformers': transformers.__version__, 'torch': torch.__version__,
    'sacrebleu': sacrebleu.__version__, 'amp': AMP,
    'bleu_tokenize': 'none',
}
manifest_path = RUN_DIR / 'contract.json'
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != contract:
        raise ValueError('Run configuration/data changed. Use a new RUN_NAME; do not mix checkpoints.')
else:
    manifest_path.write_text(json.dumps(contract, indent=2), encoding='utf-8')


In [ ]:
tokenizer = MarianTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)


def encode_pairs(pairs, tokenizer, source_limit, target_limit):
    encoded = []
    src_cut = tgt_cut = 0
    for offset in range(0, len(pairs), 256):
        chunk = pairs[offset:offset + 256]
        # Count truncation on the untruncated token lengths first.
        src = tokenizer([s for s, _ in chunk], truncation=False)['input_ids']
        tgt = tokenizer(text_target=[t for _, t in chunk], truncation=False)['input_ids']
        src_cut += sum(len(ids) > source_limit for ids in src)
        tgt_cut += sum(len(ids) > target_limit for ids in tgt)
        # Tokenize with truncation to preserve the model's terminal EOS correctly.
        features = tokenizer([s for s, _ in chunk], max_length=source_limit, truncation=True)
        labels = tokenizer(text_target=[t for _, t in chunk], max_length=target_limit,
                           truncation=True)['input_ids']
        encoded.extend({'input_ids': ids, 'attention_mask': mask, 'labels': label}
                       for ids, mask, label in zip(features['input_ids'], features['attention_mask'], labels))
    print(f'Truncated training pairs: source={src_cut}/{len(pairs)}, target={tgt_cut}/{len(pairs)}')
    return encoded


train_features = encode_pairs(train_pairs, tokenizer, MAX_SOURCE_TOKENS, MAX_TARGET_TOKENS)
# Original valid_pairs stay untouched: evaluation NEVER decodes truncated training labels as references.


In [ ]:
def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(temporary, path)


def checkpoint_from_pointer(run_dir, pointer='latest'):
    path = Path(run_dir) / 'checkpoint-index.json'
    if not path.exists():
        return None
    names = json.loads(path.read_text(encoding='utf-8')).get(pointer, [])
    if not names:
        return None
    for name in names:
        candidate = Path(run_dir) / name
        if (candidate / 'complete.json').exists():
            return candidate
    raise RuntimeError(f'No complete checkpoint behind {path}; do not silently restart training')


def publish_checkpoint(run_dir, name, is_best=False):
    path = Path(run_dir) / 'checkpoint-index.json'
    selections = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    previous = checkpoint_from_pointer(run_dir)
    selections['latest'] = [name] + ([previous.name] if previous is not None else [])
    if is_best:
        selections['best'] = [name]
    # One filesystem commit publishes both selections, never a higher best score
    # in latest with the previous best weights still selected.
    write_json_atomic(path, selections)
    return selections


def save_checkpoint(model, tokenizer, optimizer, scaler, state, run_dir, local_dir,
                    contract, is_best=False):
    start = time.monotonic()
    rng = {'python': random.getstate(), 'numpy': np.random.get_state(),
           'torch': torch.get_rng_state(),
           'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    local = Path(local_dir) / 'checkpoint-staging'
    if local.exists():
        shutil.rmtree(local)
    local.mkdir(parents=True)
    model.save_pretrained(local, safe_serialization=True)
    tokenizer.save_pretrained(local)
    torch.save({'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'state': dict(state), 'rng': rng, 'contract': contract}, local / 'training.pt')
    name = f'checkpoint-{state["step"]:08d}-{uuid.uuid4().hex[:8]}'
    destination = Path(run_dir) / name
    # Keep the previous complete snapshot while copying. Publish the marker LAST.
    shutil.copytree(local, destination)
    write_json_atomic(destination / 'complete.json', {'step': state['step'], 'contract': contract})
    selections = publish_checkpoint(run_dir, name, is_best)
    retained = {name for names in selections.values() for name in names}
    # Remove only this run's obsolete completed snapshots; leave interrupted copies identifiable.
    for old in Path(run_dir).glob('checkpoint-*'):
        if old.name not in retained and (old / 'complete.json').exists():
            shutil.rmtree(old)
    shutil.rmtree(local)
    return time.monotonic() - start


def restore_training(checkpoint, optimizer, scaler, contract):
    # training.pt includes RNG objects; load only checkpoints created by this notebook.
    payload = torch.load(Path(checkpoint) / 'training.pt', map_location='cpu', weights_only=False)
    if payload['contract'] != contract:
        raise ValueError('Checkpoint contract does not match this run')
    optimizer.load_state_dict(payload['optimizer'])
    scaler.load_state_dict(payload['scaler'])
    rng = payload['rng']
    random.setstate(rng['python'])
    np.random.set_state(rng['numpy'])
    torch.set_rng_state(rng['torch'])
    if rng['cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng['cuda'])
    return payload['state']


In [ ]:
@torch.inference_mode()
def translate_sources(model, tokenizer, sources, batch_size, source_limit, target_limit,
                      num_beams=4, deadline=None):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    offset = 0
    slowest_batch = 0.0
    while offset < len(sources):
        if deadline is not None and time.monotonic() + max(5.0, slowest_batch * 1.5) >= deadline:
            return None  # An incomplete validation must never compete for best checkpoint.
        start = time.monotonic()
        inputs = tokenizer(sources[offset:offset + batch_size], padding=True, truncation=True,
                           max_length=source_limit, return_tensors='pt').to(device)
        try:
            ids = model.generate(**inputs, num_beams=num_beams, max_new_tokens=target_limit,
                                 do_sample=False, use_cache=True)
        except torch.cuda.OutOfMemoryError:
            del inputs
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print('Reducing inference batch size to', batch_size)
            continue
        predictions.extend(tokenizer.batch_decode(ids.cpu(), skip_special_tokens=True))
        offset += inputs['input_ids'].shape[0]
        del ids, inputs
        slowest_batch = max(slowest_batch, time.monotonic() - start)
    return predictions


def evaluate_translation(model, tokenizer, valid_pairs, batch_size, source_limit,
                         target_limit, num_beams=4, deadline=None):
    predictions = translate_sources(model, tokenizer, [s for s, _ in valid_pairs], batch_size,
                                     source_limit, target_limit, num_beams, deadline)
    if predictions is None:
        return None
    references = [t for _, t in valid_pairs]  # Full original text, including long references.
    metric = sacrebleu.metrics.BLEU(tokenize='none')
    score = metric.corpus_score(predictions, [references])
    return {'bleu': score.score, 'signature': str(metric.get_signature()),
            'samples': len(references), 'predictions': predictions}


def export_submission(sources, predictions, output_dir, split='public_test'):
    if len(sources) != len(predictions):
        raise ValueError('Prediction count does not match test rows')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = split.removesuffix('_test') + '_submission'
    csv_path, zip_path = output_dir / f'{stem}.csv', output_dir / f'{stem}.zip'
    pd.DataFrame({'tieng_trung': sources, 'tieng_viet': predictions}).to_csv(
        csv_path, index=False, encoding='utf-8-sig')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname=csv_path.name)
    return csv_path, zip_path


In [ ]:
resume_checkpoint = checkpoint_from_pointer(RUN_DIR)
if resume_checkpoint is not None:
    # Stage once from Drive; do not train using a directory on Drive.
    local_resume = LOCAL_DIR / 'resume'
    if local_resume.exists():
        shutil.rmtree(local_resume)
    shutil.copytree(resume_checkpoint, local_resume)
    model = MarianMTModel.from_pretrained(local_resume).to(DEVICE)
    tokenizer = MarianTokenizer.from_pretrained(local_resume)
else:
    model = MarianMTModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE)
model.config.use_cache = False
collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda', enabled=AMP)
state = {'epoch': 0, 'next_batch': 0, 'step': 0, 'batch_size': None,
         'best_bleu': None, 'pretrained_bleu': None, 'history': []}
if resume_checkpoint is not None:
    state = restore_training(local_resume, optimizer, scaler, contract)
    print('Resuming:', resume_checkpoint.name, 'epoch:', state['epoch'], 'next batch:', state['next_batch'])
print('Parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
def calibrate_batch(model, collator, features, scaler, candidates=(128,64,32,16, 8, 4, 2, 1)):
    device = next(model.parameters()).device
    if device.type != 'cuda':
        raise RuntimeError('Batch calibration needs CUDA')
    # Probe independent longest encoder/decoder sequences, not merely a short first batch.
    longest_src = max(features, key=lambda x: len(x['input_ids']))
    longest_tgt = max(features, key=lambda x: len(x['labels']))
    example = dict(longest_src, labels=longest_tgt['labels'])
    rng = torch.cuda.get_rng_state_all()
    cpu_rng = torch.get_rng_state()
    model.train()
    try:
        for size in candidates:
            batch = loss = reserve = None
            try:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                # Account for Adam's two float32 moment buffers before optimizer's first step.
                reserve = torch.empty(2 * sum(p.numel() for p in model.parameters()),
                                      dtype=torch.float32, device=device)
                batch = {k: v.to(device) for k, v in collator([dict(example) for _ in range(size)]).items()}
                start = time.monotonic()
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    loss = model(**batch).loss
                scaler.scale(loss).backward()
                torch.cuda.synchronize()
                seconds = time.monotonic() - start
                peak = torch.cuda.max_memory_allocated()
                memory = torch.cuda.get_device_properties(device).total_memory
                if peak < 0.80 * memory:
                    print(f'Pilot: batch={size}, {seconds:.2f}s forward/backward, peak={peak / 2**30:.2f} GiB')
                    return size
            except torch.cuda.OutOfMemoryError:
                pass
            finally:
                model.zero_grad(set_to_none=True)
                del batch, loss, reserve
                gc.collect()
                torch.cuda.empty_cache()
        raise RuntimeError('Even batch=1 does not fit with memory headroom')
    finally:
        torch.set_rng_state(cpu_rng)
        torch.cuda.set_rng_state_all(rng)


def train_with_budget(model, tokenizer, features, valid_pairs, optimizer, scaler, state,
                      collator, run_dir, local_dir, contract, seconds=10800,
                      reserve_seconds=600, save_every_seconds=600, max_epochs=10):
    if seconds <= reserve_seconds or reserve_seconds < 0:
        raise ValueError('Training budget must exceed checkpoint reserve')
    device = next(model.parameters()).device
    start = time.monotonic()
    deadline = start + seconds - reserve_seconds
    last_save = start
    slowest_step = 0.0
    save_duration = 0.0
    batch_size = state['batch_size']
    if batch_size is None:
        batch_size = calibrate_batch(model, collator, features, scaler)
        state['batch_size'] = batch_size
    batches_per_epoch = math.ceil(len(features) / batch_size)
    print('Batch size:', batch_size, 'batches/epoch:', batches_per_epoch,
          'working budget minutes:', round((seconds - reserve_seconds) / 60, 1))

    def persist(is_best=False):
        nonlocal last_save, save_duration
        save_duration = max(save_duration, save_checkpoint(
            model, tokenizer, optimizer, scaler, state, run_dir, local_dir, contract, is_best))
        last_save = time.monotonic()

    # Establish a recoverable initial model and optimizer before consuming the budget.
    if checkpoint_from_pointer(run_dir) is None:
        persist(is_best=state['best_bleu'] is not None)
    epoch_loss = 0.0
    epoch_steps = 0
    while state['epoch'] < max_epochs:
        # A dedicated generator avoids changing dropout RNG when reconstructing a resumed loader.
        order = torch.randperm(len(features), generator=torch.Generator().manual_seed(
            contract['seed'] + state['epoch'])).tolist()
        remaining = order[state['next_batch'] * batch_size:]
        loader = DataLoader([features[i] for i in remaining], batch_size=batch_size,
                            collate_fn=collator, num_workers=0,
                            generator=torch.Generator().manual_seed(contract['seed']))
        model.train()
        for batch in loader:
            if time.monotonic() + max(5.0, slowest_step * 1.5, save_duration) >= deadline:
                persist()
                print('Stopping within reserved saving window; run this cell again to resume.')
                return state
            step_start = time.monotonic()
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=scaler.is_enabled()):
                loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite training loss; last complete checkpoint remains usable')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            epoch_steps += 1
            state['step'] += 1
            state['next_batch'] += 1
            del loss, batch
            if device.type == 'cuda':
                torch.cuda.synchronize()
            slowest_step = max(slowest_step, time.monotonic() - step_start)
            if state['step'] % 100 == 0:
                print(f'step={state["step"]}, epoch={state["epoch"] + 1}, '
                      f'loss={epoch_loss / epoch_steps:.4f}, '
                      f'elapsed={(time.monotonic() - start) / 60:.1f} min')
            if time.monotonic() - last_save >= save_every_seconds:
                persist()
        # Save at the epoch boundary BEFORE a potentially expensive validation.
        # Keep next_batch at the end until validation finishes: an interrupted evaluation
        # will be repeated on resume, rather than silently skipping checkpoint selection.
        persist()
        result = evaluate_translation(model, tokenizer, valid_pairs, batch_size,
                                      contract['source_limit'], contract['target_limit'],
                                      contract['beams'], deadline=deadline - max(5.0, save_duration))
        if result is None:
            print('Validation postponed: not enough budget. Resume to finish this epoch evaluation.')
            return state
        improved = state['best_bleu'] is None or result['bleu'] > state['best_bleu']
        state['history'].append({'epoch': state['epoch'] + 1, 'step': state['step'],
                                 'bleu': result['bleu'], 'signature': result['signature']})
        if improved:
            state['best_bleu'] = result['bleu']
        print(f'Epoch {state["epoch"] + 1}: BLEU={result["bleu"]:.2f}, best={state["best_bleu"]:.2f}')
        state['epoch'] += 1
        state['next_batch'] = 0
        persist(is_best=improved)
        epoch_loss = 0.0
        epoch_steps = 0
    print('Reached configured epoch limit.')
    return state


In [ ]:
# Evaluate once, on the SAME full-reference validation used during fine-tuning.
if state['pretrained_bleu'] is None:
    pretrained_result = evaluate_translation(model, tokenizer, valid_pairs, 8,
                                             MAX_SOURCE_TOKENS, MAX_TARGET_TOKENS, NUM_BEAMS)
    state['pretrained_bleu'] = pretrained_result['bleu']
    state['best_bleu'] = pretrained_result['bleu']
    print('Pretrained validation BLEU:', round(pretrained_result['bleu'], 2))
    print('Metric:', pretrained_result['signature'])
    print('Evaluation examples (keep these outputs private):')
    for (source, reference), prediction in zip(valid_pairs[:5], pretrained_result['predictions'][:5]):
        print({'source': source, 'reference': reference, 'prediction': prediction})
    save_checkpoint(model, tokenizer, optimizer, scaler, state, RUN_DIR, LOCAL_DIR, contract, is_best=True)
else:
    print('Recorded pretrained BLEU:', state['pretrained_bleu'])


In [ ]:
# Each invocation grants a new 3-hour training budget; total across sessions can exceed 3 hours.
state = train_with_budget(model, tokenizer, train_features, valid_pairs, optimizer, scaler,
                          state, collator, RUN_DIR, LOCAL_DIR, contract,
                          seconds=TRAIN_SECONDS, reserve_seconds=SAVE_RESERVE_SECONDS,
                          save_every_seconds=SAVE_EVERY_SECONDS, max_epochs=MAX_EPOCHS)


In [ ]:
# Full final evaluation is OUTSIDE the training time budget.
# Also allow a partially trained epoch to compete, instead of always falling back to pretrained.
final_result = evaluate_translation(model, tokenizer, valid_pairs, state['batch_size'] or 8,
                                    MAX_SOURCE_TOKENS, MAX_TARGET_TOKENS, NUM_BEAMS)
improved = state['best_bleu'] is None or final_result['bleu'] > state['best_bleu']
state['history'].append({'event': 'session-end', 'step': state['step'],
                         'bleu': final_result['bleu'], 'signature': final_result['signature']})
if improved:
    state['best_bleu'] = final_result['bleu']
save_checkpoint(model, tokenizer, optimizer, scaler, state, RUN_DIR, LOCAL_DIR, contract, is_best=improved)
summary = {'pretrained_bleu': state['pretrained_bleu'], 'latest_bleu': final_result['bleu'],
           'best_bleu': state['best_bleu'], 'metric_signature': final_result['signature'],
           'validation_samples': len(valid_pairs), 'step': state['step'], 'history': state['history']}
write_json_atomic(RUN_DIR / 'metrics.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
best_checkpoint = checkpoint_from_pointer(RUN_DIR, 'best')
if best_checkpoint is None:
    raise RuntimeError('No evaluated best checkpoint. Run the evaluation cells first.')
# Offload training weights, retaining the original model/optimizer for a later training invocation.
model.to('cpu')
gc.collect()
torch.cuda.empty_cache()
local_best = LOCAL_DIR / 'best-inference'
if local_best.exists():
    shutil.rmtree(local_best)
local_best.mkdir()
# Inference does not need optimizer state. Copy only model/tokenizer files.
for file in best_checkpoint.iterdir():
    if file.is_file() and file.name not in ('training.pt', 'complete.json'):
        shutil.copy2(file, local_best / file.name)
submission_model = MarianMTModel.from_pretrained(local_best).to(DEVICE)
submission_tokenizer = MarianTokenizer.from_pretrained(local_best)
try:
    predictions = translate_sources(submission_model, submission_tokenizer, test_src,
                                    state['batch_size'] or 8, MAX_SOURCE_TOKENS,
                                    MAX_TARGET_TOKENS, NUM_BEAMS)
    # Truyền 'nlp' thay vì TEST_SPLIT để xuất ra file nlp_submission.csv và nlp_submission.zip
    csv_path, zip_path = export_submission(test_src, predictions, SUBMISSION_DIR, 'nlp')
    print('Selected checkpoint:', best_checkpoint.name, 'BLEU:', state['best_bleu'])
    print('CSV:', csv_path, 'ZIP:', zip_path, 'rows:', len(predictions))
finally:
    del submission_model
    gc.collect()
    torch.cuda.empty_cache()
    model.to(DEVICE)

## Cách tiếp tục và đọc kết quả

- Phiên mới: mở notebook, giữ nguyên `ROOT`, `RUN_NAME` và cấu hình, rồi Run All.
  `checkpoint-index.json` cập nhật đồng thời lựa chọn `latest` để resume và `best` theo validation BLEU.
- Cùng phiên: có thể chạy lại cell training để cấp thêm 3 giờ, sau đó chạy lại final evaluation/submission.
- Đổi dữ liệu, model, tokenizer limits, seed, LR hoặc version: đổi `RUN_NAME`; không trộn optimizer cũ.
- Drive cần khoảng **4–5 GiB trống** cho snapshots/optimizer. Giữ latest, previous latest và best;
  một bản copy bị ngắt trước marker vẫn có thể chiếm chỗ, nhưng không được chọn để resume.
- Thời gian 3 giờ không bao gồm cài đặt/download, đánh giá ban đầu, final evaluation hoặc inference.
  Training dừng sớm 10 phút để lưu; Drive chậm có thể làm vượt giới hạn mềm.
- Source/target train giới hạn 256 token; notebook in tỷ lệ bị cắt. Validation references không bị cắt.
  Input inference vẫn bị cắt theo source limit, output giới hạn 256 token; kiểm tra câu dài trong kết quả.
- `metrics.json` ghi pretrained/latest/best BLEU và chữ ký SacreBLEU. Đây không phải điểm leaderboard.
  GRU baseline cũ dùng split/reference khác: không so trực tiếp hai con số BLEU đó.
  BLEU dùng `tokenize='none'`: chấm các token phân cách bằng khoảng trắng, không detokenize.
  Run mặc định mới là `marian-zh-vi-none-v1`; không dùng điểm best/history cũ của `13a`.
- Nếu BLEU fine-tuned thấp hơn pretrained, notebook chọn pretrained đã được đánh giá trên validation này.
- CSV giữ `tieng_trung`, `tieng_viet`, UTF-8 BOM, không index; ZIP chỉ có CSV cùng tên.
  Cell submission xuất `nlp_submission.csv` và `nlp_submission.zip` cho bộ test được chọn.
  Xác nhận format/metric với ban tổ chức trước khi nộp; không tự đổi `_` hoặc tokenization tiếng Việt.
- Không commit checkpoint/dataset hoặc output chứa dữ liệu/predictions của notebook.

### Nguồn
- [Model card, ngôn ngữ và giấy phép](https://huggingface.co/Helsinki-NLP/opus-mt-zh-vi)
- [Marian trong Transformers 4.57.3](https://huggingface.co/docs/transformers/v4.57.3/en/model_doc/marian)
- [Fine-tune translation và dynamic padding](https://huggingface.co/docs/transformers/v4.57.3/en/tasks/translation)
